# Nepali parliament ASR benchmark (Kaggle)

Reference-backed Phase 0 benchmark for the resumable transcription pipeline.

**Kaggle settings**
- Accelerator: GPU T4 x2 or P100
- Internet: ON
- Outputs: `/kaggle/working`

**Data**
- `kiranpantha/NepaliParliamentDSv1`, test split.
- The dataset has Devanagari references but no declared license. Use it for evaluation only; do not redistribute the audio or references.

This notebook prepares 100 duration-stratified held-out clips, runs WER/CER for four models, and writes a ZIP.


## 1. Check the environment


In [ ]:
import ctypes.util
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
for path in ["/kaggle/working", "/root/.cache", "/tmp"]:
    try:
        usage = shutil.disk_usage(path)
        print(f"{path}: {usage.free / 1024**3:.1f} GiB free")
    except FileNotFoundError:
        print(f"{path}: missing")

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)
print("GPU:", gpu.stdout.strip() or gpu.stderr.strip() or "not detected")
if gpu.returncode != 0:
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")


## 2. Clone the benchmark branch and install dependencies


In [ ]:
REPO_URL = "https://github.com/DarpanAdhikari/youtube-caption.git"
BRANCH = "phase0-benchmark"
REPO_DIR = Path("/kaggle/working/youtube-caption")

if not REPO_DIR.exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

need_ffmpeg = shutil.which("ffmpeg") is None
need_sndfile = ctypes.util.find_library("sndfile") is None
if need_ffmpeg or need_sndfile:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "ffmpeg", "libsndfile1"],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-pipeline.txt"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "datasets", "soundfile"],
    check=True,
)

print("Repository:", REPO_DIR)
print(
    subprocess.run(
        ["ffmpeg", "-version"],
        capture_output=True,
        text=True,
    ).stdout.splitlines()[0]
)


## 3. Prepare held-out parliament clips

Only the test parquet is downloaded. Audio bytes are decoded with `soundfile`, which avoids the newer `datasets` TorchCodec requirement. Clips shorter than 1.5 seconds and references with fewer than two words are discarded. The remaining clips are sampled across the duration distribution.


In [ ]:
import io
from collections import Counter

import numpy as np
import soundfile as sf
from datasets import Audio, load_dataset
from huggingface_hub import hf_hub_download

DATASET_REPO = "kiranpantha/NepaliParliamentDSv1"
PARQUET_FILE = "data/test-00000-of-00001.parquet"
CLIP_LIMIT = 100
BENCH_ROOT = Path("/kaggle/working/benchmark")
RUN_DIR = BENCH_ROOT / "parliament-100"
AUDIO_DIR = RUN_DIR / "audio"
REF_DIR = RUN_DIR / "references"
for path in (AUDIO_DIR, REF_DIR):
    path.mkdir(parents=True, exist_ok=True)

parquet_path = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        repo_type="dataset",
        filename=PARQUET_FILE,
    )
)
dataset = load_dataset(
    "parquet",
    data_files={"test": str(parquet_path)},
    split="test",
)
dataset = dataset.cast_column("audio", Audio(decode=False))
print("Raw test rows:", len(dataset))

rows = []
decode_failures = 0
for row in dataset:
    text = (row.get("sentence") or "").strip()
    audio = row.get("audio") or {}
    payload = audio.get("bytes")
    if not payload or not text:
        continue
    try:
        array, sampling_rate = sf.read(
            io.BytesIO(payload),
            dtype="float32",
            always_2d=False,
        )
    except Exception:
        decode_failures += 1
        continue
    if getattr(array, "ndim", 0) > 1:
        array = array.mean(axis=1)
    sampling_rate = int(sampling_rate)
    if sampling_rate <= 0:
        continue
    duration = len(array) / sampling_rate
    if duration < 1.5 or len(text.split()) < 2:
        continue
    rows.append(
        {
            "utterance_id": row["utterance_id"],
            "batch_id": row.get("batch_id", ""),
            "text": text,
            "duration": duration,
            "sampling_rate": sampling_rate,
            "array": np.asarray(array),
        }
    )

rows.sort(key=lambda item: item["duration"])
if len(rows) > CLIP_LIMIT:
    step = len(rows) / CLIP_LIMIT
    selected = [
        rows[min(len(rows) - 1, int(index * step))]
        for index in range(CLIP_LIMIT)
    ]
else:
    selected = rows

del dataset
print("Decode failures:", decode_failures)

manifest = []
for index, row in enumerate(selected, start=1):
    stem = f"clip_{index:04d}"
    sf.write(
        AUDIO_DIR / f"{stem}.wav",
        row["array"],
        row["sampling_rate"],
        subtype="PCM_16",
    )
    (REF_DIR / f"{stem}.txt").write_text(row["text"] + "\n", encoding="utf-8")
    manifest.append(
        {
            "clip": stem,
            "utterance_id": row["utterance_id"],
            "batch_id": row["batch_id"],
            "duration": round(row["duration"], 3),
            "sampling_rate": row["sampling_rate"],
            "reference": row["text"],
        }
    )

(RUN_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
durations = [item["duration"] for item in manifest]
print(f"Prepared {len(manifest)} clips")
print(
    "Duration: "
    f"mean={np.mean(durations):.2f}s "
    f"median={np.median(durations):.2f}s "
    f"max={max(durations):.2f}s"
)
print("Sampling rates:", Counter(item["sampling_rate"] for item in manifest))
for item in manifest[:3]:
    print(item["clip"], item["duration"], item["reference"])


## 4. Run the reference-backed benchmark


In [ ]:
from transcript_pipeline.benchmark import benchmark_models
from transcript_pipeline.config import PipelineConfig

MODELS = [
    "nepali-medium-ct2",
    "nepali-turbo-ct2",
    "openai-large-v3",
    "nepali-large-v3",
]
RESULTS_DIR = RUN_DIR / "results"
MODEL_CACHE = Path("/root/.cache/transcript_pipeline/models")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

audio_paths = sorted(AUDIO_DIR.glob("*.wav"))
references = {
    path.stem: (REF_DIR / f"{path.stem}.txt").read_text(encoding="utf-8")
    for path in audio_paths
}
config = PipelineConfig(
    language="ne",
    device="auto",
    compute_type="float16",
    model_cache_dir=MODEL_CACHE,
)
print(f"{len(audio_paths)} clips x {len(MODELS)} models")
summary = benchmark_models(
    MODELS,
    audio_paths,
    references,
    config,
    RESULTS_DIR,
)


def pct(value):
    return "n/a" if value is None else f"{value:.2%}"


for aggregate in summary["aggregates"]:
    if aggregate.get("load_error"):
        print(f"{aggregate['model']}: LOAD FAILED: {aggregate['load_error']}")
        continue
    print(
        f"{aggregate['model']}: "
        f"clips={aggregate['clips']} scored={aggregate['scored_clips']} "
        f"mean WER={pct(aggregate['mean_wer'])} "
        f"median WER={pct(aggregate['median_wer'])} "
        f"mean CER={pct(aggregate['mean_cer'])} "
        f"median CER={pct(aggregate['median_cer'])} "
        f"RTF={aggregate['rtf']}"
    )

import pandas as pd

pd.DataFrame(summary["rows"]).to_csv(RESULTS_DIR / "rows.csv", index=False)
for model in MODELS:
    scored = [
        row
        for row in summary["rows"]
        if row["model"] == model and row["wer"] is not None
    ]
    worst = sorted(scored, key=lambda row: row["wer"], reverse=True)[:5]
    if worst:
        print(f"\nWorst {model} clips:")
        for row in worst:
            print(
                row["audio"].split("/")[-1],
                f"WER={row['wer']:.2%}",
                f"CER={row['cer']:.2%}",
            )

print("\nSummary:", RESULTS_DIR / "summary.md")


## 5. Package the results


In [ ]:
archive = shutil.make_archive(
    str(BENCH_ROOT / "parliament-100-results"),
    "zip",
    root_dir=str(RUN_DIR),
)
print("Archive:", archive)
print("Download it from the Kaggle Output panel.")
print()
print((RESULTS_DIR / "summary.md").read_text(encoding="utf-8"))


## Notes

- If CUDA runs out of memory, rerun the benchmark cell with `compute_type="int8_float16"`.
- `nepali-large-v3` is converted to CT2 on first use; the download and conversion can take several minutes.
- This benchmark measures short, in-domain parliament utterances. It does not test chunking, resume, or long-form drift; use the 600-second sample and a full recording after the gate.
- The dataset license is undeclared. Keep the audio and references for evaluation only.
